# Deterministic Linear Inference

## The Continuous Problem

We consider the problem of inferring an unknown model parameter $m$ from noisy observations $\mathbf{\tilde{d}}$, where $m$ lies in a separable Hilbert space $\mathcal{M}$ and $\mathbf{\tilde{d}} \in \mathcal{D}$, a finite-dimensional Euclidean data space $\mathcal{D} \cong \mathbb{R}^{N_d}$ ($N_d \in \mathbb{N}$ is the number of data).

The relationship between the model and the data is described by a forward operator:
$$
G: \mathcal{M} \to \mathcal{D}, \quad [G(m)]_i = \langle K_i, m \rangle_{\mathcal{M}}
$$
assumed to be linear and bounded. The observed data is modeled as:
$$
\mathbf{\tilde{d}} = G(m) + \bm{\eta},
$$
where the unknown perturbation $\bm{\eta}$ is not modeled probabilistically, but is instead assumed to belong to a deterministic data-confidence set $\mathcal{V} \subset \mathcal{D}$.

To regularize this ill-posed inverse problem, we use a deterministic prior set in model space. Rather than a Gaussian prior measure, we assume the true model lies in a closed convex set
$$
m \in \mathcal{B} \subset \mathcal{M},
$$
which encodes prior information such as bounded energy, bounded misfit from a reference model, or other deterministic constraints.


The goal of deterministic linear inference is to characterize the admissible property set of the true model. Let 
$$
\mathcal{T} \colon \mathcal{M} \to \mathcal{P}, \quad [\mathcal{T}(m)]_i = \langle T_i, m \rangle_{\mathcal{M}}
$$
be a linear and bounded mapping that takes in a model and outputs a finite number of real scalars that represent some local or global property of the model, such as a global average, a local average, a local gradient average, or a Fourier basis coefficient. The property space $\mathcal{P}$ is a finite dimensional Hilbert space equivalent to $\mathbb{R}^{N_p}$ where $N_p \in \mathbb{N}$ is the number of properties to be sought.


The admissible property set is
$$
\mathcal{U} = \mathcal{T}\left(\mathcal{B} \cap G^{-1}(\mathbf{\tilde{d}} - \mathcal{V})\right).
$$

## Continuous Solution

For each direction $q \in \mathcal{P}$, the support function of $\mathcal{U}$ is given by the master dual equation
$$
h_{\mathcal{U}}(q) = \inf_{\lambda \in \mathcal{D}} \left\{ \langle \lambda, \mathbf{\tilde{d}} \rangle_{\mathcal{D}} + \sigma_{\mathcal{B}}(T^* q - G^* \lambda) + \sigma_{\mathcal{V}}(-\lambda) \right\},
$$
where $\sigma_{\mathcal{B}}$ and $\sigma_{\mathcal{V}}$ are the support functions of the model prior set and data-confidence set. By evaluating this support in the directions $\pm e_i$, we obtain lower and upper bounds for each property component.

In [ ]:
from intervalinf import IntervalDomain, Lebesgue, Function
from intervalinf import IntegrationConfig, ParallelConfig, LebesgueIntegrationConfig
from intervalinf.providers import NormalModesProvider, BumpFunctionProvider
from intervalinf.operators import SOLAOperator

from pygeoinf import EuclideanSpace
from pygeoinf.convex_analysis import BallSupportFunction
from pygeoinf.convex_optimisation import PrimalKKTSolver

import matplotlib.pyplot as plt
import numpy as np

# Figures are displayed interactively but not written to disk.

## Integration and Parallel Configuration

Configure numerical integration and parallelization settings for all operators in the notebook.

In [ ]:
# =============================================================================
# INTEGRATION AND PARALLEL CONFIGURATION
# =============================================================================

# Integration configuration for Lebesgue space
# - method: Integration method ('simpson', 'trapz', 'quad')
# - n_points: Number of integration points
Lebesgue_integration_cfg = LebesgueIntegrationConfig(
    inner_product=IntegrationConfig(method='simpson', n_points=500),
    dual=IntegrationConfig(method='simpson', n_points=500),
    general=IntegrationConfig(method='simpson', n_points=500)
)

# Integration configuration for SOLA operators (G and T)
sola_integration_cfg = IntegrationConfig(
    method='simpson',
    n_points=1000
)

# Parallel configuration for dense matrix computations
parallel_cfg = ParallelConfig(
    enabled=True,
    n_jobs=1  # Keep this maintained demo resource-bounded
)

print("Configuration loaded:")
print(f"  SOLA integration: {sola_integration_cfg.method}, {sola_integration_cfg.n_points} points")
print(f"  Parallel processing: {'enabled' if parallel_cfg.enabled else 'disabled'}, {parallel_cfg.n_jobs} jobs")

## Creating the spaces

The model space is the basis-free Lebesgue $L^2$ space on the interval $[0,1]$, with the ordinary inner product:

$$\langle f, g \rangle = \int_0^1 f(x) g(x) \, dx. $$

This keeps the demonstration focused on deterministic linear inference in a continuous function space.

In [ ]:
# Create a function domain and spaces
function_domain = IntervalDomain(0, 1)
N = 0
M = Lebesgue(N, function_domain, basis=None,
             integration_config=Lebesgue_integration_cfg,
             parallel_config=parallel_cfg,)
N_d = 50 # number of data points
D = EuclideanSpace(N_d) # data space
N_p = 20 # number of property points
P = EuclideanSpace(N_p) # property space

x = function_domain.uniform_mesh(1000)
print('Spaces created successfully')

In [ ]:
# Create forward and property mappings
width = 0.2 # width of the bump target functions
centers = np.linspace(function_domain.a + width / 2, function_domain.b - width / 2, N_p) # centers of the bumps

# Create a normal modes provider for the forward operator
# and a bump function provider for the target operator
# Note: The random_state is set to ensure reproducibility of results

normal_modes_provider = NormalModesProvider(
        M,
        n_modes_range=(1, 50),
        coeff_range=(-5, 5),
        gaussian_width_percent_range=(1, 5),
        freq_range=(0.1, 20),
        random_state=2,
    )

G = SOLAOperator(
    M,
    D,
    kernels=normal_modes_provider,
    cache_kernels=True,
    integration_config=sola_integration_cfg,
)

target_provider = BumpFunctionProvider(M, centers=centers, default_width=width)
T = SOLAOperator(
    M,
    P,
    kernels=target_provider,
    cache_kernels=True,
    integration_config=sola_integration_cfg,
)

print('Operators created successfully')
print(f"G cache enabled: {G.get_cache_info()['caching_enabled']}")
print(f"T cache enabled: {T.get_cache_info()['caching_enabled']}")

## Visualizing the kernels

The first two figures are intentionally kept stylistically identical to the PLI notebook. They show the sensitivity kernels used by the forward operator and the bump kernels used by the property operator.

In [ ]:
# Publication-quality Sensitivity Kernels figure

import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted", color_codes=True)
plt.figure(figsize=(12, 4), dpi=200)

# Plot all kernels in a single color - overlapping areas show regions of high sensitivity
for i in range(N_d):
    plt.plot(
        x,
        G.get_kernel(i).evaluate(x),
        color='tab:blue',
        alpha=0.35,
        linewidth=1.2
    )

plt.title(r"Sensitivity Kernels $K_i(x)$", fontsize=18)
plt.xlabel(r"$x$", fontsize=16)
plt.ylabel("Kernel Value", fontsize=16)
plt.tight_layout()
plt.grid(True, linestyle=':', alpha=0.4)
sns.despine()

plt.show()

In [ ]:
# Publication-quality Target Kernels figure

plt.figure(figsize=(12, 4), dpi=200)
for i in range(N_p):
    plt.plot(
        x,
        T.get_kernel(i).evaluate(x),
        color='tab:orange',
        alpha=0.6,
        linewidth=1.5
    )

plt.title(r"Target Kernels $T_i(x)$", fontsize=18)
plt.xlabel(r"$x$", fontsize=16)
plt.ylabel("Kernel Value", fontsize=16)
plt.tight_layout()
plt.grid(True, linestyle=':', alpha=0.4)
sns.despine()

plt.show()

## Synthetic Data Generation

### True Model Construction

We create the same synthetic true model as in the PLI notebook:

$$\bar{m}(x) = \exp\left(-\frac{(x - 0.5)^2}{0.5^2}\right) \sin(5\pi x) + x$$

From this true model, we generate synthetic observations $\mathbf{\bar{d}} = G(\bar{m})$ and add noise to create realistic measurements $\mathbf{\tilde{d}}$. The observed data are therefore identical to the PLI notebook, but in DLI the perturbation is treated as belonging to a deterministic confidence ball instead of a Gaussian law.

In [ ]:
# Create the synthetic true model
m_bar = Function(M, evaluate_callable=lambda x: np.exp(-((x - function_domain.center)/0.5)**2) * np.sin(5 * np.pi * x) + x)

# Publication-quality true model plot

plt.figure(figsize=(12, 4), dpi=200)
plt.plot(x, m_bar.evaluate(x), color='tab:red', linewidth=2.5, label=r'$\bar{m}(x)$')
plt.title(r"True Model $\bar{m}(x)$", fontsize=18)
plt.xlabel(r"$x$", fontsize=16)
plt.ylabel("Model Value", fontsize=16)
plt.legend(fontsize=14)
plt.grid(True, linestyle=':', alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

# Generate synthetic observations
print("Generating synthetic data...")
d_bar = G(m_bar)  # Clean observations
noise_level = 0.1 * np.max(d_bar)
np.random.seed(42)  # For reproducibility
d_tilde = d_bar + np.random.normal(0, noise_level, d_bar.shape)  # Noisy observations

print(f"Signal-to-noise ratio: {np.max(d_bar) / noise_level:.1f}")
print(f"Number of observations: {len(d_tilde)}")

# Publication-quality data comparison plot

plt.figure(figsize=(12, 4), dpi=200)
data_indices = np.arange(len(d_bar))

# Plot connection lines between true and noisy data
for i in range(len(d_bar)):
    plt.plot([i, i], [d_bar[i], d_tilde[i]], color='gray', alpha=0.3, linewidth=0.8)

# Plot the data points
plt.scatter(data_indices, d_tilde, label='Noisy Observations',
           color='tab:blue', alpha=0.7, marker='o', s=25, edgecolors='white', linewidths=0.5)
plt.scatter(data_indices, d_bar, label='True Data',
           color='tab:red', alpha=0.8, marker='x', s=30, linewidths=1.5)

plt.xlabel('Observation Index', fontsize=16)
plt.ylabel('Data Value', fontsize=16)
plt.title('Synthetic Observations: Truth vs. Noisy Measurements', fontsize=18)
plt.legend(fontsize=14)
plt.grid(True, linestyle=':', alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Deterministic data-confidence set and model prior set
noise_vector = d_tilde - d_bar
data_radius = 1.05 * np.linalg.norm(noise_vector)
m_0 = Function(M, evaluate_callable=lambda x: x)
model_radius = 1.05 * M.norm(M.subtract(m_bar, m_0))

print(f"Deterministic data radius: {data_radius:.4f}")
print(f"Deterministic model radius: {model_radius:.4f}")

# Visualize observed data with realized perturbation bars
mean_values = d_tilde
err_values = np.abs(noise_vector)


plt.figure(figsize=(12, 4), dpi=200)
data_indices = np.arange(len(mean_values))

plt.scatter(data_indices, mean_values, label='Observed Data', color='tab:blue', alpha=0.8, s=30)
plt.errorbar(data_indices, mean_values, yerr=err_values, fmt='none',
            color='tab:blue', alpha=0.5, capsize=2, capthick=1,
            label='Realized Perturbation Magnitude')

plt.title("Deterministic Data Confidence: Observations and Perturbations", fontsize=18)
plt.xlabel('Observation Index', fontsize=16)
plt.ylabel('Data Value', fontsize=16)
plt.legend(fontsize=14)
plt.grid(True, linestyle=':', alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

model_prior_support = BallSupportFunction(M, m_0, model_radius)
data_error_support = BallSupportFunction(D, D.zero, data_radius)

In [ ]:
# Property-prior bounds induced by the deterministic model set
prior_qs = [P.basis_vector(i) for i in range(P.dim)]
prior_upper = np.array([model_prior_support(T.adjoint(q)) for q in prior_qs])
prior_lower = np.array([
    -model_prior_support(M.multiply(-1.0, T.adjoint(q))) for q in prior_qs
])
prior_mid = 0.5 * (prior_lower + prior_upper)
prior_half = 0.5 * (prior_upper - prior_lower)
true_props = T(m_bar)


plt.figure(figsize=(12, 5), dpi=200)

plt.errorbar(centers, prior_mid, yerr=prior_half, fmt='o', color='tab:blue',
            alpha=0.7, capsize=4, capthick=2, markersize=6,
            label='Property Prior (center ±2 half-widths)')
plt.fill_between(centers, prior_mid - prior_half, prior_mid + prior_half,
                color='tab:blue', alpha=0.15)

plt.scatter(centers, true_props, label='True Properties',
           color='tab:red', marker='x', s=100, alpha=0.9, linewidths=3, zorder=10)

mean_prop_prior = T(m_0)
plt.plot(centers, mean_prop_prior, color='tab:green', linewidth=2.5, label='Prior Center Property', zorder=9)

plt.xlabel('Target Location', fontsize=16)
plt.ylabel('Property Value', fontsize=16)
plt.title('Property Prior: Deterministic Bounds Before Data', fontsize=18)
plt.legend(fontsize=14)
plt.grid(True, linestyle=':', alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

print(f"Prior width range: [{(prior_upper - prior_lower).min():.3f}, {(prior_upper - prior_lower).max():.3f}]")

In [ ]:
# Deterministic linear inference via PrimalKKTSolver
#
# The solver works directly in abstract Hilbert-space vectors -
# the model space is never discretised.  For each direction q in
# property space, we:
#   1. Form  c = T*(q)  in model space H
#   2. Solve  max <c, u>_H  s.t. u in B, G(u) in d_tilde - V
#      via the Woodbury-KKT system (only an M x M Cholesky solve)
#   3. Extract the bound value  <c, u*>_H

import time
solve_t0 = time.perf_counter()

kkt_solver = PrimalKKTSolver(
    model_prior_support,
    data_error_support,
    G,
    d_tilde,
 )

basis_directions = [P.basis_vector(i) for i in range(P.dim)]
neg_basis_directions = [P.multiply(-1.0, P.basis_vector(i)) for i in range(P.dim)]

upper_results = [kkt_solver.solve(T.adjoint(q)) for q in basis_directions]
lower_results = [kkt_solver.solve(T.adjoint(q)) for q in neg_basis_directions]

upper_bounds = np.array([
    M.inner_product(T.adjoint(q), r.m)
    for q, r in zip(basis_directions, upper_results)
 ])
lower_bounds = -np.array([
    M.inner_product(T.adjoint(q), r.m)
    for q, r in zip(neg_basis_directions, lower_results)
 ])

solve_elapsed = time.perf_counter() - solve_t0

p_tilde = 0.5 * (lower_bounds + upper_bounds)
std_P_post = 0.5 * (upper_bounds - lower_bounds)

plt.figure(figsize=(12, 6), dpi=200)

plt.errorbar(centers, p_tilde, yerr=std_P_post, fmt="o", color="tab:blue",
            alpha=0.8, capsize=4, capthick=2, markersize=8, linewidth=2,
            label="Admissible Properties (center ± half-widths)")

plt.fill_between(centers, p_tilde - std_P_post, p_tilde + std_P_post,
                color="tab:blue", alpha=0.2)

plt.scatter(centers, true_props, label="True Properties",
           color="tab:red", marker="x", s=120, alpha=0.9, linewidths=4, zorder=10)

plt.plot(centers, mean_prop_prior, "o--", color="tab:green", alpha=0.6,
        markersize=6, linewidth=2, label="Prior Center Properties")

plt.xlabel("Target Location", fontsize=16)
plt.ylabel("Property Value", fontsize=16)
plt.title("Property Inference Results (DLI — PrimalKKT)", fontsize=18)
plt.legend(fontsize=14, loc="best")
plt.grid(True, linestyle=":", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

property_errors = np.abs(p_tilde - true_props)
within_bounds = np.sum((true_props >= lower_bounds) & (true_props <= upper_bounds))
all_results = upper_results + lower_results
total_kkt_iterations = sum(r.num_iterations for r in all_results)
n_converged = sum(r.converged for r in all_results)
g_cache_info = G.get_cache_info()
t_cache_info = T.get_cache_info()

print("\n" + "=" * 50)
print("FINAL RESULTS SUMMARY")
print("="*50)
print("Workflow: deterministic admissible-property bounds on the continuous model space")
print("Solver: Primal KKT (Woodbury, no model-space matrix)")
print(f"Wall-clock solve time: {solve_elapsed:.2f} s")
print(f"Properties captured by admissible intervals: {within_bounds}/{len(true_props)} ({100*within_bounds/len(true_props):.1f}%)")
print(f"Mean absolute error of interval centers: {np.mean(property_errors):.4f}")
print(f"RMS error of interval centers: {np.sqrt(np.mean(property_errors**2)):.4f}")
print(f"Max interval half-width: {np.max(std_P_post):.4f}")
print(f"Average interval half-width: {np.mean(std_P_post):.4f}")
print(f"KKT solves converged: {n_converged}/{len(all_results)}")
print(f"Total fsolve evaluations: {total_kkt_iterations}")
print(f"G mesh built: {g_cache_info['shared_mesh_built']}, eval-cache entries: {g_cache_info.get('kernel_eval_cache_entries', 0)}")
print(f"T mesh built: {t_cache_info['shared_mesh_built']}, eval-cache entries: {t_cache_info.get('kernel_eval_cache_entries', 0)}")
print("="*50)